In [ ]:
conda config --add channels defaults
conda config --add channels conda-forge
conda config --add channels bioconda
conda config --set channel_priority strict


config = configuration（配置）
去这几个库里找所需的包

In [ ]:
conda install -c bioconda -c conda-forge trim-galore cutadapt fastqc


-c 是 --channel 的缩写，表示“从哪个软件源（仓库）找包”。

In [ ]:
zcat C5_1.fq.gz | awk 'NR==2{print length($0); exit}'
zcat C5_2.fq.gz | awk 'NR==2{print length($0); exit}'


zcat C5_1.fq.gz

C5_1.fq.gz 是 gzip 压缩的 FASTQ 文件（R1）。

zcat 会把这个压缩文件解压并把内容输出到屏幕/管道（不生成解压后的新文件）

awk 'NR==2{print length($0); exit}'

awk 是按“行”处理文本的工具。

NR：当前处理到的行号（从 1 开始）

NR==2：只在第 2 行时执行花括号 { ... } 里的动作

length($0)：计算当前这一整行的字符长度

$0 表示“当前整行文本”

print length($0)：打印该行长度

exit：立刻退出 awk

为什么取第 2 行？

FASTQ 每条 read 占 4 行：

第1行：@read_id（read 名称）

第2行：ACTG...（序列）

第3行：+（分隔符）

第4行：IIII...（质量值）

In [ ]:
mkdir -p /root/autodl-tmp/RRBS/C5/trim_out trim_galore --paired --rrbs --quality 20 --length 30 --output_dir /root/autodl-tmp/RRBS/C5/trim_out C5_1.fq.gz C5_2.fq.gz

In [ ]:
mkdir -p /root/autodl-tmp/RRBS/C5/trim_out 

mkdir：创建目录

-p：递归创建。如果上级目录 /root/autodl-tmp/RRBS/C5 不存在，会一并创建；如果目录已存在也不会报错。

In [ ]:
trim_galore --paired --rrbs --quality 20 --length 30 

trim_galore

这是一个“封装脚本”，内部主要调用：

cutadapt：去接头 + 质量剪切

--paired

表示你输入的是双端测序（paired-end），也就是：

C5_1.fq.gz = R1

C5_2.fq.gz = R2

它会把两端 reads 成对处理，并输出成对的结果。
-rrbs

告诉 Trim Galore：这是 RRBS 文库（常见为 MspI 消化产生的 CCGG 片段）。

RRBS 的特性是：

插入片段（insert）经常很短 → 很容易读穿进入 adapter → 接头污染更常见。

某些建库步骤会造成 read 末端偏倚，--rrbs 会做 RRBS 的特异处理来减少这种偏倚（比如在特定位置额外剪几 bp）。

--quality 20

质量剪切阈值（Phred score）为 20。

含义：

在 reads 的末端（通常是 3’ 端），如果出现低质量碱基，会被剪掉。

Q20 大致表示错误率约 1%（经验尺度），是生信里非常常用的保守阈值。

--length 30

设定最短长度阈值：trimming 后长度 < 30 bp 的 reads 会被过滤掉。

为什么要设这个？

太短的 reads 很难唯一比对（容易多重比对/随机比对）。

对 RRBS 来说，太短会影响后续比对和 CpG 覆盖质量。

为什么你用 30 比 50 更稳？

你是 PE150 RRBS，短插入会造成大量 reads 被剪短。

直接 50 可能“误杀”很多；30 是更常见的起步折中。


In [ ]:
--output_dir /root/autodl-tmp/RRBS/C5/trim_out 

指定输出目录到数据盘。

In [ ]:
C5_1.fq.gz C5_2.fq.gz

In [ ]:
输入文件

WARNING: adapter sequences may be incomplete
用于剪切的 adapter 序列可能不是“全长接头”，Trim Galore 经常只用 Illumina 接头的前 12–13bp

RRBS reads trimmed by additional 2 bp when adapter contamination was detected:	51444132 (91.2%)
有 91.2% 检测到了 read-through/adapter contamination，因此触发了 RRBS 的“额外 2bp”剪切。
读取片段长度固定，但插入的DNA短，所以把接头也读进去了

In [ ]:
/root/autodl-tmp/RRBS/C5/trim_out/C5_1.fq.gz_trimming_report.txt:Total reads processed:              56,404,004
/root/autodl-tmp/RRBS/C5/trim_out/C5_1.fq.gz_trimming_report.txt:Reads written (passing filters):    56,404,004 (100.0%)
/root/autodl-tmp/RRBS/C5/trim_out/C5_1.fq.gz_trimming_report.txt:Total written (filtered):  6,618,626,940 bp (78.2%)
/root/autodl-tmp/RRBS/C5/trim_out/C5_2.fq.gz_trimming_report.txt:Total reads processed:              56,404,004
/root/autodl-tmp/RRBS/C5/trim_out/C5_2.fq.gz_trimming_report.txt:Reads written (passing filters):    56,404,004 (100.0%)
/root/autodl-tmp/RRBS/C5/trim_out/C5_2.fq.gz_trimming_report.txt:Total written (filtered):  6,623,343,856 bp (78.3%)

“filtered” 和 “passing filters”一个是reads 条数是否被保留，另一个是碱基（bp）总量经过剪切后的剩余量。
passing filters = 100% 就是：所有 reads 剪完之后长度都 ≥30bp（没有 read 因太短被丢弃）

In [ ]:
fastqc -o /root/autodl-tmp/RRBS/C5/trim_out /root/autodl-tmp/RRBS/C5/trim_out/*val_*.fq.gz


-o 是 output directory（输出目录） 参数。
/root/autodl-tmp/RRBS/C5/trim_out/：路径

*val_*.fq.gz：通配符（wildcard）

conda install -c bioconda -c conda-forge bismark bowtie2 samtools fastqc multiqc -y

A. bismark：甲基化测序比对 + 甲基化位点提取（核心）

用途：专门用于亚硫酸氢盐测序（BS-seq：RRBS/WGBS）

将 reads 比对到参考基因组（实际调用 bowtie2）

从比对结果中提取每个 CpG/CHG/CHH 位点的甲基化状态

生成比对报告、甲基化报告等

典型输出：*_bismark_bt2*.bam、*_report.txt、CpG_context*.txt.gz 等

B. bowtie2：短序列比对程序（被 bismark 调用）

用途：通用的 DNA reads 比对工具（非甲基化专用）。
在 bismark 里，它负责真正的“对齐”，bismark 负责“把 BS 转换的特殊情况处理好 + 甲基化判定”。

你什么时候直接用它？
做普通 DNA-seq/RNA-seq（非 BS）比对时会直接用；做 RRBS/WGBS 时通常是 bismark 间接调用。

C. samtools：BAM/SAM 文件处理的“瑞士军刀”

用途：对齐结果通常是 SAM/BAM 格式，samtools 用来：

查看/统计：samtools flagstat、samtools stats

排序：samtools sort

建索引：samtools index

筛选：按 mapping quality、按是否成对、是否重复等过滤

在 bismark 流程里的地位：几乎必备，处理比对后的 BAM 文件非常常用。

D. fastqc：单样本质控（每个 fastq 生成一个报告）

用途：对原始 reads 或修剪后的 reads 做 QC：

读长分布、质量分布

GC 含量

接头污染、过度代表序列（overrepresented sequences）

duplication level 等

什么时候用：

Trimming 前做一次（看原始问题）

Trimming 后再做一次（确认修剪效果）

E. multiqc：汇总多个样本/多个软件的报告（项目级）

用途：把一堆 fastqc、bismark、samtools 等输出的日志/报告自动汇总成一个总报告（HTML），方便你一眼看全项目的结果。

典型用法：在结果目录里运行 multiqc .，它会自动搜集并生成 multiqc_report.html。

In [ ]:
mkdir -p /root/refgenome/sika_deer

bismark_genome_preparation --bowtie2 /root/refgenome/sika_deer

bismark_genome_preparation
Bismark 自带的“参考基因组准备工具”。

--bowtie2
指定后续比对用 Bowtie2，因此这里会调用 bowtie2-build 去建索引。
（如果不用这个参数，可能会用 Bowtie1 或默认策略，取决于你的安装/版本）

/root/refgenome/sika_deer
指定“基因组目录”。这个目录里必须有你的 genome.fa（或多个 fasta）。

Bismark 的思路是：

把参考基因组做 C→T、G→A 等方向的转换，生成“转换版参考序列”

对这些转换版序列分别用 Bowtie2 建索引

比对时，把 reads 也做相应转换，再去这些索引里找最佳匹配

最后再回推每个 C 位点到底是“C（甲基化）还是 T（未甲基化）

In [ ]:
mkdir -p /root/autodl-tmp/ref/sika_deer
ln -sf /root/autodl-tmp/RRBS/C5/sika_deer_reference.fna /root/autodl-tmp/ref/sika_deer/genome.fa

ln：创建链接（link）

-s：创建符号链接（symlink，类似 Windows 的快捷方式）

-f：强制覆盖
如果目标位置已经存在同名文件/链接（genome.fa），就直接替换掉，不会提示你确认。

/root/autodl-tmp/RRBS/C5/sika_deer_reference.fna
这是你实际的参考基因组 FASTA 文件

/root/autodl-tmp/ref/sika_deer/genome.fa
这会变成一个“指向源文件的链接”，名字叫 genome.fa。

In [ ]:
screen -S bismark_index

screen
一个终端复用器（terminal multiplexer）：在同一台服务器上管理多个“可脱离”的终端会话。

-S bismark_index
-S 用来给新会话命名。这里会话名就是 bismark_index，方便你之后查找/重新连接。

In [ ]:
mkdir -p /root/autodl-tmp/RRBS/C5/bismark_out

bismark --bowtie2 --genome /root/autodl-tmp/ref/sika_deer \
  -1 /root/autodl-tmp/RRBS/C5/trim_out/C5_1_val_1.fq.gz \
  -2 /root/autodl-tmp/RRBS/C5/trim_out/C5_2_val_2.fq.gz \
  --rrbs -p 12 --bam --sort \
  -o /root/autodl-tmp/RRBS/C5/bismark_out

Bismark 是 **亚硫酸氢盐测序（BS-seq / RRBS）**常用流程工具：

把 reads 以“C→T / G→A 转换”的方式进行比对

在 BAM 里附带甲基化相关标签（如 XM/XR/XG），并输出比对统计报告

--genome /root/autodl-tmp/ref/sika_deer

指定参考基因组目录。这个目录必须同时包含：

未转换的参考基因组 fasta（如 genome.fa）

bismark_genome_preparation 生成的 Bisulfite_Genome 子目录（CT/GA 转换索引）
官方说明里明确：基因组目录需要同时包含原始 fasta 和准备步骤生成的两个 bisulfite 子目录。
--bowtie2

选择底层比对器用 Bowtie2。

-1 ...C5_1_val_1.fq.gz 和 -2 ...C5_2_val_2.fq.gz

-1：成对测序的 read1（R1）

-2：成对测序的 read2（R2）

文件名里的 val_1 / val_2 通常来自 Trim Galore，表示“配对关系已校验后的输出”。


In [ ]:
bismark_methylation_extractor --paired-end --comprehensive \
  --bedGraph --cytosine_report --genome_folder /root/autodl-tmp/ref/sika_deer \
  *.deduplicated.bam

1) bismark_methylation_extractor

Bismark 的“甲基化提取/调用”步骤：从 BAM/SAM 里把每个 C 的甲基化判定写出来。官方说明输出的最基础格式是：
<seq-ID> <methylation state> <chromosome> <position> <methylation call>（1-based 坐标）

2) --paired-end（或 -p）

说明输入是 双端测序产生的 Bismark 结果文件；如果你不写，工具也会尝试自动判断，但写上更明确。

额外关键点：paired-end 的 BAM 不要做按坐标排序（coordinate sort）。提取器要求 Read1/Read2 在文件里紧挨着出现；如果发现被按染色体位置排序，会直接退出并要求提供“unsorted file”。

3) --comprehensive

把 四条链（OT/CTOT/OB/CTOB） 的甲基化信息合并到 按序列上下文（context） 的输出里：默认会生成 3 类（CpG/CHG/CHH）文件。
对比默认模式：默认会按链+上下文拆成很多文件（每个输入 BAM 可达 12 个文件）。

4) --bedGraph

在完成基础提取后，进一步把结果转成 排序后的 bedGraph（0-based start, 1-based end）并同时写出一个 coverage 文件（1-based 坐标，带甲基化/非甲基化计数）。
并且它会临时按染色体拆分文件来排序，排序完会清理临时文件——所以需要一定磁盘空间。

5) --cytosine_report

在 bedGraph/coverage 之后，再生成 genome-wide cytosine report：

默认 CpG-only（除非你再加 --CX/--CX_context）

报告会包含 全基因组每个位置（即使该位点没有 reads 覆盖，也会输出，计数为 0）

因此文件可能非常大（哺乳动物基因组可到十亿行量级）。

6) --genome_folder /root/autodl-tmp/ref/sika_deer

给 --cytosine_report 用的：它需要知道你用于比对的参考基因组位置（目录里包含 .fa/.fasta）。并且在 cytosine report 模式下这是必需的。

7) *.deduplicated.bam

这是 shell 通配符：会在“当前目录”下匹配所有以 .deduplicated.bam 结尾的文件，并把它们作为输入（提取器支持一次给多个文件）。

In [ ]:
cd /root/autodl-tmp/RRBS/C5

# 若还没有基因汇总，先生成
awk 'BEGIN{FS=OFS="\t"; print "gene_id","gene_name","strand","CpG_sites","mC","uC","total","meth_pct","status"}
NR>1{
  k=$8 OFS $9 OFS $10; n[k]++; m[k]+=$4; u[k]+=$5
}
END{
  for(k in n){
    t=m[k]+u[k]; p=(t>0?100*m[k]/t:0);
    s=(t<10?"NoCoverage":(p>=10?"Methylated":"Unmethylated"));
    print k,n[k],m[k],u[k],t,sprintf("%.2f",p),s
  }
}' CpG_with_gene.tsv | sort -t $'\t' -k8,8nr > gene_methylation_summary.tsv


 看整体甲基化水平（全样本）

In [ ]:
awk -F'\t' 'NR>1{m+=$4;u+=$5;n++} END{t=m+u; printf("sites=%d\tmC=%.0f\tuC=%.0f\toverall_meth=%.2f%%\n",n,m,u,100*m/t)}' CpG_with_gene.tsv


看注释命中率（基因内 vs 基因间）

In [ ]:
awk -F'\t' 'NR>1{tot++; if($8!="NA") ann++; else inter++}
END{printf("annotated=%d(%.2f%%)\tintergenic=%d(%.2f%%)\n",ann,100*ann/tot,inter,100*inter/tot)}' CpG_with_gene.tsv


In [ ]:
看染色体层面的甲基化

awk -F'\t' 'NR>1{m[$1]+=$4;u[$1]+=$5;n[$1]++}
END{for(c in n){t=m[c]+u[c]; printf("%s\t%d\t%.2f\n",c,n[c],100*m[c]/t)}}' CpG_with_gene.tsv | sort -k1,1 > chr_methylation.tsv
head chr_methylation.tsv


In [ ]:
高甲基化/低甲基化基因

In [ ]:
# 高甲基化基因：CpG位点>=20 且 meth_pct>=80
awk -F'\t' 'NR==1 || ($4>=20 && $8+0>=80)' gene_methylation_summary.tsv | head -n 30 > genes_high_meth.tsv

# 低甲基化基因：CpG位点>=20 且 meth_pct<=20
awk -F'\t' 'NR==1 || ($4>=20 && $8+0<=20)' gene_methylation_summary.tsv | head -n 30 > genes_low_meth.tsv
